# Jour 4 : Évaluation Rigoureuse & Mise en Production

## Phase 0 : Mise en route

Installation des packages requis (si nécessaire) et chargement des données de classification (Breast Cancer).

In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


In [4]:
X, y = load_breast_cancer(return_X_y=True)
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

valeurs, comptes = np.unique(y, return_counts=True)
for val, compte in zip(valeurs, comptes):
    pourcentage = (compte / len(y)) * 100
    print(f"Classe {val} : {compte} ({pourcentage:.1f}%)")

Dimensions de X : (569, 30)
Dimensions de y : (569,)
Classe 0 : 212 (37.3%)
Classe 1 : 357 (62.7%)


## Phase 1 : Séparer les données proprement (Train / Validation / Test)

In [5]:
def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    """Découpe X, y en trois jeux : train, validation, test.
    Doit renvoyer 6 objets : X_train, X_val, X_test, y_train, y_val, y_test.
    Les proportions doivent rester respectées (utiliser stratify=y).
    """
    # TODO : premier split pour isoler le test (test_size)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    # TODO : deuxième split sur le reste pour isoler la validation
    # TODO : penser à recalculer la proportion val sur le reste
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=val_size, random_state=random_state, stratify=y_train_val
    )
    
    # TODO : renvoyer les 6 objets
    return X_train, X_val, X_test, y_train, y_val, y_test

In [6]:
print("CAS NORMAL")
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42)
print(f"Train : {X_train.shape[0]} | Validation : {X_val.shape[0]} | Test : {X_test.shape[0]}")
print("Répartition des classes conservée dans chaque jeu : oui")

CAS NORMAL
Train : 364 | Validation : 91 | Test : 114
Répartition des classes conservée dans chaque jeu : oui


In [8]:
print("CAS LIMITE (val_size=0)")
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y, test_size=0.2, val_size=0, random_state=42)

CAS LIMITE (val_size=0)


InvalidParameterError: The 'test_size' parameter of train_test_split must be a float in the range (0.0, 1.0), an int in the range [1, inf) or None. Got 0 instead.

In [9]:
print("CAS ADVERSARIAL (dataset déséquilibré à 95/5)")
np.random.seed(42)
X_adv = np.random.randn(500, 2)
y_adv = np.array([0] * 475 + [1] * 25)

X_train_a, X_val_a, X_test_a, y_train_a, y_val_a, y_test_a = split_train_val_test(X_adv, y_adv, test_size=0.2, val_size=0.2, random_state=42)
print(f"Train : {X_train_a.shape[0]} (Classe 1 : {np.sum(y_train_a)}) | Validation : {X_val_a.shape[0]} (Classe 1 : {np.sum(y_val_a)}) | Test : {X_test_a.shape[0]} (Classe 1 : {np.sum(y_test_a)})")

CAS ADVERSARIAL (dataset déséquilibré à 95/5)
Train : 320 (Classe 1 : 16) | Validation : 80 (Classe 1 : 4) | Test : 100 (Classe 1 : 5)


## Phase 2 : Bootstrap et bagging, comprendre le rééchantillonnage

Le bootstrap consiste à tirer des échantillons avec remise pour évaluer la stabilité des performances du modèle.

In [1]:
def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42, replace=True):
    """Évalue la stabilité d'un modèle par bootstrap.
    Pour chaque itération : tirer un échantillon AVEC REMISE de même taille
    que X, entraîner, évaluer sur les points NON tirés (out-of-bag).
    Doit renvoyer la liste des scores et afficher moyenne et écart-type.
    """
    # TODO : rng = np.random.default_rng(random_state)
    rng = np.random.default_rng(random_state)
    scores = []
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    
    # TODO : pour chaque itération, tirer des indices avec remise (rng.choice avec replace=True)
    for i in range(n_iterations):
        bootstrap_indices = rng.choice(indices, size=n_samples, replace=replace)
        
        # TODO : identifier les indices out-of-bag (OOB) pour évaluer
        #         OOB = les points jamais tirés dans cet échantillon bootstrap,
        #         ils n'ont pas servi à l'entraînement : jeu de test gratuit
        oob_indices = np.setdiff1d(indices, bootstrap_indices)
        
        if len(oob_indices) == 0:
            continue
            
        # TODO : entraîner sur l'échantillon, scorer sur l'out-of-bag
        X_train_b, y_train_b = X[bootstrap_indices], y[bootstrap_indices]
        X_oob, y_oob = X[oob_indices], y[oob_indices]
        
        modele.fit(X_train_b, y_train_b)
        score = modele.score(X_oob, y_oob)
        scores.append(score)
        
    # TODO : renvoyer la liste, afficher moyenne ± écart-type
    scores = np.array(scores)
    if len(scores) > 1:
        moyenne = np.mean(scores)
        ecart_type = np.std(scores)
        print(f"Score moyen sur {len(scores)} bootstraps : {moyenne:.3f} (± {ecart_type:.3f})")
    elif len(scores) == 1:
        print(f"Score sur 1 bootstrap : {scores[0]:.3f} (pas d'écart-type possible)")
        
    return list(scores)

In [10]:
print("CAS NORMAL")
modele_lr = LogisticRegression(max_iter=10000, random_state=42)
scores_normal = bootstrap_scores(modele_lr, X, y, n_iterations=30, random_state=42)

CAS NORMAL
Score moyen sur 30 bootstraps : 0.952 (± 0.015)


In [11]:
print("CAS LIMITE (n_iterations=1)")
scores_limite = bootstrap_scores(modele_lr, X, y, n_iterations=1, random_state=42)

CAS LIMITE (n_iterations=1)
Score sur 1 bootstrap : 0.920 (pas d'écart-type possible)


In [12]:
print("CAS ADVERSARIAL (sans remise : replace=False)")
scores_adv = bootstrap_scores(modele_lr, X, y, n_iterations=30, replace=False, random_state=42)

CAS ADVERSARIAL (sans remise : replace=False)
